<a href="https://colab.research.google.com/github/brunoss-on/scimlga2026/blob/main/resources_GeracaoTiros/gera_tiros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Environment setup

import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ENV = "colab"
    print("Running on Google Colab")
else:
    ENV = "local"
    print("Running locally")

Running on Google Colab


In [2]:
if ENV == "colab":

  print("Install dependencies")
  !pip install deepwave==0.0.10

  # Library Imports
  import time
  import torch
  import numpy as np
  import scipy.ndimage
  import matplotlib.pyplot as plt
  import deepwave
  from deepwave import scalar
  import cv2
  import os
else:
   import time
   import torch
   import numpy as np
   import scipy.ndimage
   import matplotlib.pyplot as plt
   import deepwave
   from deepwave import scalar
   import cv2
   import os

Install dependencies
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 11.9 MB/s eta 0:00:00
  Created wheel for deepwave: filename=deepwave-0.0.10-py3-none-any.whl size=45139 sha256=7f0ff88058add9166ce454289d4e0d9eb1f9f20aacdd8b6a71375a02c13d6bc0
  Stored in directory: /root/.cache/pip/wheels/93/05/32/5eff15f407b7315abf53465751d83f1009dc2afae4631d6ee3
Successfully built deepwave


In [3]:
# Device Configuration: use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
# Set up the workspace
import os
REPO_DIR = "/content/scimlga2026"

if not os.path.exists(REPO_DIR):
    print("Cloning the SciMLGA 2026 course repository from GitHub...")
    !git clone https://github.com/brunoss-on/scimlga2026.git {REPO_DIR}
    print("Repository successfully cloned!")
else:
    print("Repository already available. Skipping clone.")


WORK_DIR = os.path.join(REPO_DIR, "resources_GeracaoTiros")
os.chdir(WORK_DIR)
print("Current directory:", os.getcwd())
print("Current contents:", os.listdir())

Cloning the SciMLGA 2026 course repository from GitHub...
Cloning into '/content/scimlga2026'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 48 (delta 16), reused 19 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 27.46 MiB | 28.32 MiB/s, done.
Resolving deltas: 100% (16/16), done.
Repository successfully cloned!
Current directory: /content/scimlga2026/resources_GeracaoTiros
Current contents: ['gera_tiros.ipynb']


In [6]:
# Set acquisition parameters for seismic modeling with Deepwave
dx = 10.0
dt = 0.0030
nz = 128
ny = 128
nt = 640
peak_freq = 8
peak_time = 1.0 / peak_freq
num_shots = 12
num_sources_per_shot = 1
num_receivers_per_shot = 128 # um receptor a cada cela do modelo
source_spacing = 10 # em número de celas (100m)
receiver_spacing = 1 # em número de celas (10m)
first_source= 4 # borda para acomodar os 12 tiros
first_receiver=0
source_depth=1
receiver_depth=1

In [7]:
# geometria do levantamento
# source_locations
source_locations = torch.zeros(num_shots,num_sources_per_shot,2,
                               dtype=torch.long, device=device)
source_locations[..., 1] = source_depth
source_locations[:, 0, 0] = torch.arange(num_shots)*source_spacing+ first_source

# receiver_locations
receiver_locations = torch.zeros(num_shots,num_receivers_per_shot, 2,
                                 dtype=torch.long, device=device)
receiver_locations[..., 1] = receiver_depth
receiver_locations[:, :, 0] = ((torch.arange(num_receivers_per_shot)*receiver_spacing + first_receiver)
    .repeat(num_shots, 1))

# source_amplitudes
source_amplitudes = (deepwave.wavelets.ricker(peak_freq, nt, dt,peak_time)
    .repeat(num_shots,num_sources_per_shot, 1)
    .to(device))

In [8]:
#definições dos modelos
model_homo = torch.ones(ny, nz) * 1500.0 # modelo da água para excluir a onda direta
model_homogenous=model_homo.to(device)
# Propagate homogeneous
out = scalar(model_homogenous,dx,dt,source_amplitudes=source_amplitudes,
             source_locations=source_locations,
             receiver_locations=receiver_locations,
             accuracy=8,
             pml_freq=peak_freq,pml_width=[50,50,50,50])
receiver_amplitudes_homo=out[-1] #copia as amplitudes do dado de tiro

In [9]:
# Load dataset
DATA_FILE = "Mods1.zip"
FILE_ID = "1NfYKtnY-kkhsbIAQCQ2U0CEBMMvabLGj"

DATA_DIR = os.path.join(REPO_DIR,"resources_GeracaoModelosVel")

if not os.path.exists(f"{DATA_DIR}/Mods1"):
    os.makedirs(DATA_DIR, exist_ok=True)
    !pip -q install gdown
    !gdown "https://drive.google.com/uc?id={FILE_ID}" -O "{DATA_DIR}/{DATA_FILE}"

    !unzip -q -o "{DATA_DIR}/{DATA_FILE}" -d "{DATA_DIR}"
else:
    print("Data already available.")

Downloading...
From: https://drive.google.com/uc?id=1NfYKtnY-kkhsbIAQCQ2U0CEBMMvabLGj
To: /content/scimlga2026/resources_GeracaoModelosVel/Mods1.zip
100% 1.04M/1.04M [00:00<00:00, 144MB/s]


In [12]:
!ls "{DATA_DIR}/Mods1"
!ls "{DATA_DIR}/Mods1/Bin"

Bin  Figs
model_100.vp.npy  model_213.vp.npy  model_326.vp.npy  model_439.vp.npy
model_101.vp.npy  model_214.vp.npy  model_327.vp.npy  model_43.vp.npy
model_102.vp.npy  model_215.vp.npy  model_328.vp.npy  model_440.vp.npy
model_103.vp.npy  model_216.vp.npy  model_329.vp.npy  model_441.vp.npy
model_104.vp.npy  model_217.vp.npy  model_32.vp.npy   model_442.vp.npy
model_105.vp.npy  model_218.vp.npy  model_330.vp.npy  model_443.vp.npy
model_106.vp.npy  model_219.vp.npy  model_331.vp.npy  model_444.vp.npy
model_107.vp.npy  model_21.vp.npy   model_332.vp.npy  model_445.vp.npy
model_108.vp.npy  model_220.vp.npy  model_333.vp.npy  model_446.vp.npy
model_109.vp.npy  model_221.vp.npy  model_334.vp.npy  model_447.vp.npy
model_10.vp.npy   model_222.vp.npy  model_335.vp.npy  model_448.vp.npy
model_110.vp.npy  model_223.vp.npy  model_336.vp.npy  model_449.vp.npy
model_111.vp.npy  model_224.vp.npy  model_337.vp.npy  model_44.vp.npy
model_112.vp.npy  model_225.vp.npy  model_338.vp.npy  model_450.vp.np

In [17]:
print("Current directory:", os.getcwd())
model_dir='../resources_GeracaoModelosVel/Mods1/'
tiros_dir= os.path.join(WORK_DIR,"TirosMods1/")
os.makedirs(tiros_dir, exist_ok=True)

Current directory: /content/scimlga2026/resources_GeracaoTiros


In [18]:
for nummodel in range(1,501):
#carga modelo
  model=np.load(model_dir+"/Bin/model_{}.vp.npy".format(nummodel))
  model_npy=(model.T).astype(np.float32) #rede está desenhada para esse tamanho de problema
  model =torch.ones(ny, nz) * model_npy
  model_prop=model.to(device)
  # Propagate
  out = scalar(model_prop,dx,dt,source_amplitudes=source_amplitudes,
              source_locations=source_locations,
              receiver_locations=receiver_locations,
              accuracy=8,
              pml_freq=peak_freq,pml_width=[50,50,50,50])
  receiver_amplitudes=out[-1] #copia as amplitudes do dado de tiro
  np.save(tiros_dir+'/shot_{}.npy'.format(nummodel),(receiver_amplitudes-receiver_amplitudes_homo).cpu())
  print(nummodel)


/tmp/ipykernel_1014/2386001057.py:5: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  model =torch.ones(ny, nz) * model_npy


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277


In [19]:
# for nummodel in range(1,301):
# #carga modelo
#   model=np.load("/content/drive/MyDrive/PosDoc/Curso_open/Atividade_DL/Modelos02/Bin/model_{}.vp.npy".format(nummodel))
#   model_npy=(model.T).astype(np.float32) #rede está desenhada para esse tamanho de problema
#   model =torch.ones(ny, nz) * model_npy
#   model_prop=model.to(device)
#   # Propagate
#   out = scalar(model_prop,dx,dt,source_amplitudes=source_amplitudes,
#               source_locations=source_locations,
#               receiver_locations=receiver_locations,
#               accuracy=8,
#               pml_freq=peak_freq,pml_width=[50,50,50,50])
#   receiver_amplitudes=out[-1] #copia as amplitudes do dado de tiro
#   np.save('/content/drive/MyDrive/PosDoc/Curso_open/Atividade_DL/Shots02/shot_{}.npy'.format(nummodel),(receiver_amplitudes-receiver_amplitudes_homo).cpu())
#   print(nummodel)


In [20]:
# for nummodel in range(1,301):
# #carga modelo
#   model=np.load("/content/drive/MyDrive/PosDoc/Curso_open/Atividade_DL/Modelos03/Bin/model_{}.vp.npy".format(nummodel))
#   model_npy=(model.T).astype(np.float32) #rede está desenhada para esse tamanho de problema
#   model =torch.ones(ny, nz) * model_npy
#   model_prop=model.to(device)
#   # Propagate
#   out = scalar(model_prop,dx,dt,source_amplitudes=source_amplitudes,
#               source_locations=source_locations,
#               receiver_locations=receiver_locations,
#               accuracy=8,
#               pml_freq=peak_freq,pml_width=[50,50,50,50])
#   receiver_amplitudes=out[-1] #copia as amplitudes do dado de tiro
#   np.save('/content/drive/MyDrive/PosDoc/Curso_open/Atividade_DL/Shots03/shot_{}.npy'.format(nummodel),(receiver_amplitudes-receiver_amplitudes_homo).cpu())
#   print(nummodel)
